# Imports

In [134]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [143]:
# set path so that notebook can be run from any location
import sys
import os
from pathlib import Path

# find the repository root
candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd().parent.parent,
]

repo_root = None
for p in candidates:
    if (p / "Tools").exists():
        repo_root = p
        break

if repo_root is None:
    raise FileNotFoundError("Could not find repository root containing Analysis/ and Tools/")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print("Repo root:", repo_root)
print(sys.path[:5])

sys.path.append(os.path.join(os.getcwd(), os.pardir))

Repo root: /Users/xiangxigao/Desktop/git/droplet-phenotyping
['/Users/xiangxigao/Desktop/git/droplet-phenotyping', '/Users/xiangxigao/miniforge3/envs/phenotyping/lib/python312.zip', '/Users/xiangxigao/miniforge3/envs/phenotyping/lib/python3.12', '/Users/xiangxigao/miniforge3/envs/phenotyping/lib/python3.12/lib-dynload', '']


In [144]:
from Tools.leica_tools import RawLoader, parse_lif
from Tools.sample_tools import Sample
from Tools.db_tools import DbManager

# ---- for debugging ----
import numpy as np
import pandas as pd
import os
import glob
from readlif.reader import LifFile
import datetime
from dotenv import load_dotenv


In [ ]:
# 3 regions taken, all conditions are the same, 
parse_lif('/Users/xiangxigao/Desktop/DMi8/TECH_XG_021/TECH_XG_021.lif')

,index,name,timestamp,t_index,n_channels,bit_depth,resolution,merged
0,0,1hr/Region 15 Merged,0,0,4,16,1.527160,True
1,1,1hr/Region 11 Merged,0,0,4,16,1.527160,True
2,2,1hr/Region 6 Merged,0,0,4,16,1.527159,True
3,3,2hr/Region 6 Merged,0,0,4,16,1.527160,True
4,3,2hr/Region 6 Merged,1,1,4,16,1.527160,True
5,4,2hr/Region 11 Merged,0,0,4,16,1.527159,True
6,4,2hr/Region 11 Merged,1,1,4,16,1.527159,True
7,5,2hr/Region 15 Merged,0,0,4,16,1.527160,True
8,5,2hr/Region 15 Merged,1,1,4,16,1.527160,True
9,6,4hr/Region 6 Merged,0,0,4,16,1.527160,True


# Data prep

In [149]:
expID = 'TECH_XG_021'
rawloader = RawLoader(expID)
rawloader.frame_df

,droplet_size,size_range,image_index,t_index,time,condition,path
frameID,,,,,,,
0,90,5,0,0,1,HFE7500_K562_300_150_region15,/Users/xiangxigao/Desktop/DMi8/TECH_XG_021/TEC...
1,90,5,5,0,2,HFE7500_K562_300_150_region15,/Users/xiangxigao/Desktop/DMi8/TECH_XG_021/TEC...
2,90,5,5,1,3,HFE7500_K562_300_150_region15,/Users/xiangxigao/Desktop/DMi8/TECH_XG_021/TEC...
3,90,5,8,0,4,HFE7500_K562_300_150_region15,/Users/xiangxigao/Desktop/DMi8/TECH_XG_021/TEC...
4,90,5,8,1,5,HFE7500_K562_300_150_region15,/Users/xiangxigao/Desktop/DMi8/TECH_XG_021/TEC...
5,90,5,12,0,6,HFE7500_K562_300_150_region15,/Users/xiangxigao/Desktop/DMi8/TECH_XG_021/TEC...
6,90,5,1,0,1,HFE7500_K562_300_150_region11,/Users/xiangxigao/Desktop/DMi8/TECH_XG_021/TEC...
7,90,5,4,0,2,HFE7500_K562_300_150_region11,/Users/xiangxigao/Desktop/DMi8/TECH_XG_021/TEC...
8,90,5,4,1,3,HFE7500_K562_300_150_region11,/Users/xiangxigao/Desktop/DMi8/TECH_XG_021/TEC...


# Droplet detection

Execute a preview run of the droplet detection. An image will be saved to the exp folder in the analyses directory. 
If droplets are not well detected consider changing the droplet size estimate in setup.xlsx (re-run RawLoader API).

In [148]:
frameID = 1
sample = Sample(expID, frameID)
sample.detect_droplets(mode='sweep')
sample.visualize_droplets(channel=0,save=True)

2576 droplets in frame 1 detected 



Run droplet detection through all frames of the experiment. drop_register.csv will be created at the end of the process.

In [150]:
rawloader = RawLoader(expID)
df = rawloader.frame_df.copy()

# conditions = ["HFE7500_THP1_300_170", "RAN101_THP1_300_170"]
# df = df[df["condition"].isin(conditions)].copy()
# df = df.sort_index()  # sort by the index, which is frameID

# print(df[["image_index", "t_index", "time", "condition", "path"]])

all_dfs = []
for frameID in df.index.astype(int):
    print(f"Processing frameID: {frameID}")
    sample = Sample(expID, int(frameID))
    all_dfs.append(sample.detect_droplets(mode="sweep", return_df=True))

droplets = pd.concat(all_dfs, ignore_index=True).reset_index(drop=True).rename_axis("GlobalID")

rawloader.update_droplet_df(droplets)
print("Saved droplets.csv to:", os.path.join(rawloader.exp_dir, "droplets.csv"))

Processing frameID: 0


KeyboardInterrupt: 

In [151]:
dbm = DbManager()
dbm.detect_droplets(expID, mode='sweep')

2517 droplets in frame 0 detected 

2576 droplets in frame 1 detected 

2574 droplets in frame 2 detected 

2496 droplets in frame 3 detected 

2455 droplets in frame 4 detected 

2193 droplets in frame 5 detected 

1244 droplets in frame 6 detected 

1276 droplets in frame 7 detected 

1246 droplets in frame 8 detected 

1258 droplets in frame 9 detected 

1271 droplets in frame 10 detected 

1306 droplets in frame 11 detected 

2530 droplets in frame 12 detected 

2562 droplets in frame 13 detected 

2568 droplets in frame 14 detected 

2592 droplets in frame 15 detected 

2591 droplets in frame 16 detected 

2620 droplets in frame 17 detected 



# Outlier detection

In [152]:
dbm = DbManager()
dbm.detect_outliers(expID, model_name='outlier_v3.h5')

1184/1184 ━━━━━━━━━━━━━━━━━━━━ 55s 47ms/step


2026-09-11 16:03:26.915664: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
/Users/xiangxigao/miniforge3/envs/phenotyping/lib/python3.12/contextlib.py:158: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self.gen.throw(value)
2026-09-11 16:03:30.298337: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2026-09-11 16:03:30.919206: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2026-09-11 16:03:31.009117: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2026-09-11 16:03:31.095648: W tensorflow/core/fr

In [22]:
sample = Sample(expID, 0)
sample.reload_droplets()
sample.visualize_droplets(channel=0)

# Workpackage Generation

In [3]:
dbm = DbManager()
dbm.generate_wp(expID='NKIP_FA_065', exclude_query='outlier == True')

2024-09-08 22:10:21.239744: I tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2024-09-08 22:10:21.339381: I tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2024-09-08 22:10:21.549281: I tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2024-09-08 22:10:21.961074: I tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2024-09-08 22:10:22.772556: I tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


# Cell counting

In [ ]:
dbm = DbManager()
dbm.cell_count(expID=expID, model_name='cell_count_v3.h5')

2026-09-11 17:02:25.001544: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


     73/Unknown 12s 160ms/step